# MinutiaeNet Step-by-Step Extraction Walkthrough

In this notebook, we will illustrate the didactic functioning of the MinutiaeNet model. We will walk through the loading of an image, the intermediate representations created by **CoarseNet**, and the refinement by **FineNet**, closing with the plot of the detected minutiae over the original image.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Ensure minutiaenet module can be imported

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from minutiaenet.api import postprocess
from minutiaenet.wrapper import get_minutiaenet
from minutiaenet.mnet_utils import DEFAULT_COARSENET_WEIGHTS_PATH, DEFAULT_FINENET_WEIGHTS_PATH
from minutiaenet.plot import plot_img, plot_ori_field, plot_mnt

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 8)

## 1. Setup and Loading the Image

First, we define the path to our image and load it as a grayscale numpy array normalized between 0 and 1. We will use an image from the NIST SD27 latent fingerprint dataset.

In [ ]:
image_path = "/home/joaocontreras/work/fingernet/datasets/NISTSD27/images/B101L9U.bmp"

# Load image using PIL
img_pil = Image.open(image_path).convert("L")
img_np = np.array(img_pil, dtype=np.float32) / 255.0

plt.imshow(img_np, cmap='gray')
plt.title("Original Fingerprint Input")
plt.axis('off')
plt.show()

## 2. Model Initialization

We instantiate the `MinutiaeNet` wrapper which bundles both `CoarseNet` and `FineNet`. We pass the default weights to it.

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = get_minutiaenet(
    coarsenet_weights=DEFAULT_COARSENET_WEIGHTS_PATH,
    finenet_weights=DEFAULT_FINENET_WEIGHTS_PATH,
    device=device
)
model.eval()
print("Model loaded successfully.")

## 3. CoarseNet: Feature Extraction

The first stage of the network is `CoarseNet`. It is a residual-based CNN that takes our normalized fingerprint image and internally generates fingerprint domain knowledge. It produces several outputs:
- **Orientation Field:** Describes the local orientation of the fingerprint ridges.
- **Segmentation Mask:** Identifies the region of interest separating fingerprint area from background noise.
- **Enhanced Image:** Enhances the ridge structure while suppressing noise.
- **Score Map:** A dense map indicating the probability of a minutia existing at each spatial location.

Let's pass the image through CoarseNet.

In [ ]:
# Prepare batch tensor: [Batch, Channels, Height, Width]
img_tensor = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():
    raw_outputs = model.coarsenet(img_tensor)

print("Raw CoarseNet keys:", list(raw_outputs.keys()))

## 4. FineNet and Postprocessing

To obtain the specific coordinates `(x, y)`, angle, and score strings from the raw output, we use the `postprocess` function. Internally, this function:
1. Extracts candidate patches based on peaks in the continuous `score_map` generated by CoarseNet.
2. Passes candidate patches through **FineNet** for a second stage of robust verification and minutiae orientation regression.
3. Performs Non-Maximum Suppression (NMS) to eliminate duplicate detections in the same region.

Our model is already wrapped to do this when called directly via its forward pass.

In [ ]:
with torch.no_grad():
    # Direct full forward pass via the model:
    final_outputs = model(img_tensor)
    
print("Post-processed keys:", list(final_outputs.keys()))

# Extract outputs for the first (and only) image in our batch
minutiae = final_outputs["minutiae"][0].cpu().numpy()
enhanced_img = final_outputs["enhanced_image"][0].cpu().numpy()
seg_mask = final_outputs["segmentation_mask"][0].cpu().numpy()
ori_field = final_outputs["orientation_field"][0].cpu().numpy()

print(f"Detected {len(minutiae)} minutiae.")

## 5. Visualizing the Minutiae Extraction Pipeline

Using the provided plotting script in `minutiaenet.plot`, let's visualize the step by step domain knowledge utilized by the framework alongside the final extracted minutiae. This replicates the visual representations commonly requested for fingerprint verification algorithms.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# 1. Orientation Field (Domain Knowledge)
plot_img(axes[0], ori_field)
axes[0].set_title("Orientation Field (Radians)")

# 2. Enhanced Image (Domain Knowledge)
plot_img(axes[1], enhanced_img)
axes[1].set_title("Enhanced Image")

# 3. Orientation Map Plotted over Image
stride = 16
plot_img(axes[2], img_np)
plot_ori_field(axes[2], ori_field, stride=stride)
axes[2].set_title(f"Orientation Flow (Stride: {stride})")

# 4. Final Minutiae Output over Original Image
plot_img(axes[3], img_np)
plot_mnt(axes[3], minutiae)
axes[3].set_title(f"Detected Minutiae ({len(minutiae)})")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## Conclusion

This notebook demonstrated the intermediate features (domain knowledge representations) that `CoarseNet` builds to guide its detection, and how these internal representations correlate with the final, robust minutiae points. The network successfully identified the minutiae points (represented by red squares and direction lines) overlaid over the original input latent print.